# Dealing with Imbalanced Classes: When Majority Rules

In many real-world classification problems, the classes are not evenly distributed — this is called **class imbalance**.  
Common examples: fraud detection (≈0.1% fraudulent), medical diagnosis (rare diseases), churn prediction, etc.

When one class dominates (e.g. 95% vs 5%), standard classifiers tend to be lazy: predicting the majority class every time yields 95% accuracy, but is completely useless.

This notebook covers:
- Why accuracy is misleading
- Better evaluation metrics
- Data-level methods (resampling)
- Algorithm-level methods (cost-sensitive, threshold moving)
- Ensemble methods for imbalance
- A complete pipeline with cross-validation & comparison

## 1. The Problem: Why Accuracy Is Misleading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, precision_recall_curve, roc_curve,
                             auc, matthews_corrcoef)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries imported successfully.')

In [ ]:
# Create a synthetic imbalanced dataset: 5% minority class
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=10, n_redundant=5,
    n_clusters_per_class=1, weights=[0.95, 0.05], flip_y=0.01,
    random_state=42
)

print(f'Dataset shape: {X.shape}')
print(f'Class distribution:\n{pd.Series(y).value_counts().sort_index()}')
print(f'Majority: {np.mean(y == 0):.3f}, Minority: {np.mean(y == 1):.3f}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

### Visualization 1: Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = pd.Series(y).value_counts().sort_index()
colors = ['#3498db', '#e74c3c']

axes[0].bar(['Majority (0)', 'Minority (1)'], class_counts.values,
            color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution (Entire Dataset)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=12, fontweight='bold')

axes[1].pie(class_counts.values, labels=['Majority (0)', 'Minority (1)'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            explode=(0, 0.08), shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('imgs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Dummy Classifier Shows How Accuracy Deceives

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print('=== Dummy Classifier (always predicts majority) ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_dummy):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_dummy, zero_division=0):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_dummy, zero_division=0):.4f}')
print(f'F1-score:  {f1_score(y_test, y_pred_dummy, zero_division=0):.4f}')
print(f'MCC:       {matthews_corrcoef(y_test, y_pred_dummy):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_pred_dummy):.4f}')
print()
print('The dummy gets ~95% accuracy but 0 precision/recall on minority!')

## 2. Better Metrics for Imbalanced Data

| Metric | Formula | Why It Matters |
|--------|---------|----------------|
| **Precision** | TP / (TP + FP) | Of all positive predictions, how many are correct? |
| **Recall** (Sensitivity) | TP / (TP + FN) | Of all actual positives, how many did we catch? |
| **F1 Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision & recall |
| **PR-AUC** | area under Precision-Recall curve | Sensitive to minority class performance |
| **ROC-AUC** | area under ROC curve | Less sensitive to imbalance |
| **MCC** | Matthews Correlation Coefficient | Balanced measure even for very different class sizes |

**Key insight**: For imbalanced data, **Precision-Recall curves** are more informative than ROC curves because they focus on the minority class.

In [ ]:
# Train a simple Logistic Regression as baseline
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print('=== Logistic Regression Baseline ===')
print(classification_report(y_test, y_pred_lr, target_names=['Majority', 'Minority']))
print(f'MCC:  {matthews_corrcoef(y_test, y_pred_lr):.4f}')
print(f'ROC-AUC:  {roc_auc_score(y_test, y_prob_lr):.4f}')
print(f'PR-AUC:   {average_precision_score(y_test, y_prob_lr):.4f}')

## 3. Data-Level Methods: Resampling

### 3.1 Random Undersampling
**Idea**: Randomly remove samples from the majority class until balance is achieved.  
**Pros**: Fast, reduces memory  **Cons**: Loses potentially useful information

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)

print(f'Original: {Counter(y_train)}')
print(f'Undersampled: {Counter(y_rus)}')

lr_rus = LogisticRegression(max_iter=1000, random_state=42)
lr_rus.fit(X_rus, y_rus)
y_pred_rus = lr_rus.predict(X_test)

print(f'\nTest Accuracy: {accuracy_score(y_test, y_pred_rus):.4f}')
print(f'Test F1:       {f1_score(y_test, y_pred_rus):.4f}')
print(f'Test Recall:   {recall_score(y_test, y_pred_rus):.4f}')

### 3.2 Random Oversampling
**Idea**: Randomly duplicate minority samples until balance is achieved.  
**Pros**: Simple, no information loss  **Cons**: Prone to overfitting (duplicates exact samples)

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)

print(f'Original: {Counter(y_train)}')
print(f'Oversampled: {Counter(y_ros)}')

lr_ros = LogisticRegression(max_iter=1000, random_state=42)
lr_ros.fit(X_ros, y_ros)
y_pred_ros = lr_ros.predict(X_test)

print(f'\nTest Accuracy: {accuracy_score(y_test, y_pred_ros):.4f}')
print(f'Test F1:       {f1_score(y_test, y_pred_ros):.4f}')
print(f'Test Recall:   {recall_score(y_test, y_pred_ros):.4f}')

### 3.3 SMOTE (Synthetic Minority Oversampling)

SMOTE creates **synthetic** samples rather than duplicating existing ones:
1. For each minority sample, find its k nearest minority neighbors
2. Pick a random neighbor
3. Interpolate between them: `new_sample = sample + λ × (neighbor - sample)`, λ ∈ [0, 1]

This generates realistic new samples along line segments connecting minority instances.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_train, y_train)

print(f'Original: {Counter(y_train)}')
print(f'SMOTE:    {Counter(y_sm)}')

lr_sm = LogisticRegression(max_iter=1000, random_state=42)
lr_sm.fit(X_sm, y_sm)
y_pred_sm = lr_sm.predict(X_test)

print(f'\nTest Accuracy: {accuracy_score(y_test, y_pred_sm):.4f}')
print(f'Test F1:       {f1_score(y_test, y_pred_sm):.4f}')
print(f'Test Recall:   {recall_score(y_test, y_pred_sm):.4f}')

### Visualization 2: How SMOTE Creates Synthetic Samples

In [ ]:
# Create a 2D toy dataset to visualize SMOTE
X_2d, y_2d = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, weights=[0.9, 0.1], flip_y=0.01,
    class_sep=1.2, random_state=42
)

smote_viz = SMOTE(random_state=42, k_neighbors=3)
X_2d_sm, y_2d_sm = smote_viz.fit_resample(X_2d, y_2d)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_2d[y_2d == 0, 0], X_2d[y_2d == 0, 1],
                c='#3498db', label='Majority', alpha=0.6, edgecolors='k', s=40)
axes[0].scatter(X_2d[y_2d == 1, 0], X_2d[y_2d == 1, 1],
                c='#e74c3c', label='Minority (Original)', alpha=0.8, edgecolors='k', s=60)
axes[0].set_title('Original Data (90% / 10% Split)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

orig_minority = X_2d[y_2d == 1]
smote_minority = X_2d_sm[y_2d_sm == 1]
synthetic = smote_minority[len(orig_minority):]

axes[1].scatter(X_2d[y_2d == 0, 0], X_2d[y_2d == 0, 1],
                c='#3498db', label='Majority', alpha=0.4, edgecolors='k', s=30)
axes[1].scatter(orig_minority[:, 0], orig_minority[:, 1],
                c='#e74c3c', label='Minority (Original)', alpha=0.8, edgecolors='k', s=50, marker='o')
axes[1].scatter(synthetic[:, 0], synthetic[:, 1],
                c='#2ecc71', label='SMOTE Synthetic', alpha=0.7, edgecolors='k', s=50, marker='D')

# Draw some interpolation lines for intuition
rng = np.random.RandomState(42)
for i in range(min(5, len(orig_minority))):
    idx = rng.randint(0, len(orig_minority))
    neigh_idx = rng.randint(0, len(synthetic))
    axes[1].plot([orig_minority[idx, 0], synthetic[neigh_idx, 0]],
                 [orig_minority[idx, 1], synthetic[neigh_idx, 1]],
                 'k--', alpha=0.2, lw=0.8)

axes[1].set_title('SMOTE: Synthetic Samples Along Interpolation Lines', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.savefig('imgs/smote_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Advanced SMOTE Variants & Cleaning Methods

| Method | Description |
|--------|-------------|
| **Borderline-SMOTE** | Only creates synthetic samples near the decision boundary (harder examples) |
| **ADASYN** | Adaptive Synthetic Sampling — focuses on samples harder to learn (density weighted) |
| **Tomek Links** | Removes pairs of opposite-class nearest neighbors (cleaning) |
| **Edited Nearest Neighbors** | Removes samples whose class differs from k nearest neighbors |
| **SMOTE + Tomek** | SMOTE for oversampling + Tomek for cleaning noisy samples |

In [ ]:
from imblearn.over_sampling import ADASYN, BorderlineSMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours

methods = {
    'Borderline-SMOTE': BorderlineSMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'SMOTE+Tomek': SMOTETomek(random_state=42),
}

results_adv = {}
for name, method in methods.items():
    X_res, y_res = method.fit_resample(X_train, y_train)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_res, y_res)
    y_pred = clf.predict(X_test)
    results_adv[name] = {
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Samples': len(y_res)
    }
    print(f'{name:20s} → F1={results_adv[name]["F1"]:.4f}  '
          f'Recall={results_adv[name]["Recall"]:.4f}  '
          f'Precision={results_adv[name]["Precision"]:.4f}  '
          f'Samples={results_adv[name]["Samples"]}')

## 4. Algorithm-Level Methods

### 4.1 Class Weights (Balanced Mode)

Most sklearn classifiers support `class_weight='balanced'` which automatically adjusts weights inversely proportional to class frequencies:  
`weight_j = n_samples / (n_classes × n_j)`

In [ ]:
classifiers = {
    'LR (no weight)': LogisticRegression(max_iter=1000, random_state=42),
    'LR (balanced)': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'RF (no weight)': RandomForestClassifier(random_state=42),
    'RF (balanced)': RandomForestClassifier(class_weight='balanced', random_state=42),
    'DT (balanced)': DecisionTreeClassifier(class_weight='balanced', random_state=42),
}

results_wt = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    results_wt[name] = {
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
    }
    print(f'{name:20s} → F1={results_wt[name]["F1"]:.4f}  '
          f'Recall={results_wt[name]["Recall"]:.4f}  '
          f'Precision={results_wt[name]["Precision"]:.4f}  '
          f'ROC-AUC={results_wt[name]["ROC-AUC"]:.4f}')

### 4.2 Threshold Moving

Default decision threshold is 0.5, but for imbalanced data this is rarely optimal.  
We can **tune the threshold** using the Precision-Recall curve to find the best trade-off.

In [ ]:
# Get probabilities from the baseline LR model
y_prob = lr.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-12)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'Best threshold (max F1): {best_threshold:.4f}')
print(f'Threshold=0.5 F1: {f1_score(y_test, y_prob >= 0.5):.4f}')
print(f'Threshold={best_threshold:.4f} F1: {f1_score(y_test, y_prob >= best_threshold):.4f}')
print(f'Threshold={best_threshold:.4f} Recall: {recall_score(y_test, y_prob >= best_threshold):.4f}')
print(f'Threshold={best_threshold:.4f} Precision: {precision_score(y_test, y_prob >= best_threshold):.4f}')

### Visualization 3: PR Curve with Optimal Threshold

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(recalls, precisions, color='#2c3e50', lw=2.5, label='PR Curve')
ax.scatter(recalls[best_idx], precisions[best_idx], color='#e74c3c', s=150,
           zorder=5, label=f'Optimal Threshold = {best_threshold:.3f}',
           edgecolors='k', linewidth=2)
ax.axhline(y=precisions[best_idx], color='#e74c3c', linestyle='--', alpha=0.3)
ax.axvline(x=recalls[best_idx], color='#e74c3c', linestyle='--', alpha=0.3)

pr_auc = auc(recalls, precisions)
ax.fill_between(recalls, precisions, alpha=0.15, color='#2c3e50')
ax.text(0.7, 0.5, f'PR-AUC = {pr_auc:.3f}', fontsize=14,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='yellow', alpha=0.4))
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curve with Optimal Threshold', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('imgs/pr_curve_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Focal Loss (Conceptual)

Focal Loss, introduced in the RetinaNet paper, modifies cross-entropy to focus on hard-to-classify examples:

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- When $p_t$ is large (easy sample), the modulating factor $(1-p_t)^\gamma$ is small → down-weights easy samples
- When $p_t$ is small (hard sample), the factor is large → focuses training on minority class
- γ controls the down-weighting rate (typical value: 2)

Scikit-learn doesn't natively support focal loss, but custom implementations can be used with neural networks.

## 5. Ensemble Methods for Imbalanced Data

These combine resampling with ensemble learning:

In [ ]:
from imblearn.ensemble import (BalancedRandomForestClassifier,
                               EasyEnsembleClassifier,
                               BalancedBaggingClassifier)
from imblearn.ensemble import RUSBoostClassifier
from sklearn.metrics import make_scorer

ensemble_models = {
    'BalancedRF': BalancedRandomForestClassifier(
        n_estimators=100, random_state=42, sampling_strategy='all'),
    'EasyEnsemble': EasyEnsembleClassifier(
        n_estimators=10, random_state=42),
    'RUSBoost': RUSBoostClassifier(
        n_estimators=50, random_state=42),
    'BalancedBagging': BalancedBaggingClassifier(
        n_estimators=50, random_state=42),
}

results_ens = {}
for name, clf in ensemble_models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    results_ens[name] = {
        'F1': f1_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
    }
    print(f'{name:20s} → F1={results_ens[name]["F1"]:.4f}  '
          f'Recall={results_ens[name]["Recall"]:.4f}  '
          f'Precision={results_ens[name]["Precision"]:.4f}')

## 6. Cost-Sensitive Learning

Instead of changing data distribution, assign different **misclassification costs**:
- False Negative (missing a positive) is usually far more expensive than False Positive
- Example: missing a fraudulent transaction costs $100, flagging a legitimate one costs $5

A cost matrix encodes the penalty for each type of error. The classifier is trained to minimize **total cost** rather than total error.

In [ ]:
# Cost-sensitive approach: custom class weights representing cost ratio
# Suppose FN cost is 10x FP cost
cost_ratio = 10
weights = {0: 1.0, 1: cost_ratio}

lr_cost = LogisticRegression(max_iter=1000, class_weight=weights, random_state=42)
lr_cost.fit(X_train, y_train)
y_pred_cost = lr_cost.predict(X_test)

print('=== Cost-Sensitive Learning (FN cost = 10× FP) ===')
print(f'F1:        {f1_score(y_test, y_pred_cost):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_cost):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_cost):.4f}')

# Compare different cost ratios
cost_ratios = [1, 2, 5, 10, 20, 50]
print('\n--- Cost Ratio Sensitivity ---')
for cr in cost_ratios:
    clf = LogisticRegression(max_iter=1000, class_weight={0: 1.0, 1: cr}, random_state=42)
    clf.fit(X_train, y_train)
    y_p = clf.predict(X_test)
    print(f'Ratio {cr:3d} → F1={f1_score(y_test, y_p):.4f}  '
          f'Recall={recall_score(y_test, y_p):.4f}  '
          f'Precision={precision_score(y_test, y_p):.4f}')

## 7. Anomaly Detection for Extreme Imbalance

When minority class is <1%, supervised methods struggle.  
**Anomaly/Outlier detection** treats the minority as "anomalies":
- **Isolation Forest**: isolates anomalies via random partitioning
- **One-Class SVM**: finds a boundary around the majority class
- **Elliptic Envelope**: assumes Gaussian distribution
- **LOF** (Local Outlier Factor): density-based

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor

# Create dataset with extreme imbalance (1% minority)
X_extreme, y_extreme = make_classification(
    n_samples=2000, n_features=10, n_informative=5, n_redundant=2,
    n_clusters_per_class=1, weights=[0.99, 0.01], flip_y=0.01,
    class_sep=1.5, random_state=42
)
X_train_ext, X_test_ext, y_train_ext, y_test_ext = train_test_split(
    X_extreme, y_extreme, test_size=0.3, random_state=42, stratify=y_extreme
)

print(f'Extreme imbalance: {np.mean(y_extreme == 1)*100:.2f}% minority')

anomaly_models = {
    'Isolation Forest': IsolationForest(contamination=0.015, random_state=42),
    'One-Class SVM': OneClassSVM(nu=0.015, kernel='rbf', gamma='auto'),
    'Elliptic Envelope': EllipticEnvelope(contamination=0.015, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.015, novelty=True),
}

for name, model in anomaly_models.items():
    if name == 'LOF':
        model.fit(X_train_ext)
        y_pred = model.predict(X_test_ext)
    else:
        model.fit(X_train_ext)
        y_pred = model.predict(X_test_ext)
    # Convert: anomaly detectors output 1 for inliers, -1 for outliers
    y_pred_bin = (y_pred == -1).astype(int)
    f1 = f1_score(y_test_ext, y_pred_bin)
    rec = recall_score(y_test_ext, y_pred_bin)
    prec = precision_score(y_test_ext, y_pred_bin)
    print(f'{name:20s} → F1={f1:.4f}  Recall={rec:.4f}  Precision={prec:.4f}')

## 8. Complete Pipeline: Cross-Validated Comparison

Let's compare all methods systematically using stratified cross-validation:

In [ ]:
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipelines = {
    'Baseline LR': Pipeline([
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Undersampling': ImbPipeline([
        ('sampler', RandomUnderSampler(random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Oversampling': ImbPipeline([
        ('sampler', RandomOverSampler(random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'SMOTE': ImbPipeline([
        ('sampler', SMOTE(random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'SMOTE+Tomek': ImbPipeline([
        ('sampler', SMOTETomek(random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Balanced Weights': Pipeline([
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'Balanced RF': ImbPipeline([
        ('clf', BalancedRandomForestClassifier(n_estimators=100, random_state=42, sampling_strategy='all'))
    ]),
}

scoring = {
    'F1': make_scorer(f1_score),
    'ROC-AUC': 'roc_auc',
    'Precision': make_scorer(precision_score),
    'Recall': make_scorer(recall_score),
}

results_cv = {}
for name, pipe in pipelines.items():
    print(f'Evaluating {name}...')
    scores = {}
    for score_name, scorer in scoring.items():
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=scorer)
        scores[score_name] = cv_scores
    results_cv[name] = scores

print('\n=== Cross-Validation Results (mean ± std) ===')
summary_data = []
for name in results_cv:
    row = {'Method': name}
    for score_name in ['F1', 'ROC-AUC', 'Precision', 'Recall']:
        mean = results_cv[name][score_name].mean()
        std = results_cv[name][score_name].std()
        row[f'{score_name} (mean)'] = f'{mean:.4f}'
        row[f'{score_name} (std)'] = f'{std:.4f}'
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
display(summary_df)

### Visualization 4: PR Curve vs ROC Curve Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

methods_plot = {
    'LR Baseline': lr,
    'LR + SMOTE': lr_sm,
    'Balanced RF': ensemble_models['BalancedRF'],
}

for name, model in methods_plot.items():
    if hasattr(model, 'predict_proba'):
        y_prob_m = model.predict_proba(X_test)[:, 1]
    else:
        y_prob_m = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        continue

    fpr, tpr, _ = roc_curve(y_test, y_prob_m)
    roc_auc_val = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc_val:.3f})')

    prec, rec, _ = precision_recall_curve(y_test, y_prob_m)
    pr_auc_val = auc(rec, prec)
    axes[1].plot(rec, prec, lw=2, label=f'{name} (AUC={pr_auc_val:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves (More Informative for Imbalance)',
                  fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('imgs/pr_vs_roc.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualization 5: Methods Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

methods_names = list(results_cv.keys())
f1_means = [results_cv[m]['F1'].mean() for m in methods_names]
f1_stds = [results_cv[m]['F1'].std() for m in methods_names]
recall_means = [results_cv[m]['Recall'].mean() for m in methods_names]
prec_means = [results_cv[m]['Precision'].mean() for m in methods_names]

x = np.arange(len(methods_names))
width = 0.25

bars1 = ax.bar(x - width, f1_means, width, yerr=f1_stds,
               label='F1 Score', color='#3498db', capsize=4, edgecolor='black')
bars2 = ax.bar(x, recall_means, width,
               label='Recall', color='#2ecc71', edgecolor='black')
bars3 = ax.bar(x + width, prec_means, width,
               label='Precision', color='#e74c3c', edgecolor='black')

ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Cross-Validated Performance: Imbalanced Classification Methods',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods_names, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim([0, 1])
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)

for bar, val in zip(bars1, f1_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('imgs/methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualization 6: Decision Boundary With & Without SMOTE

In [ ]:
def plot_decision_boundary(X, y, model, ax, title):
    h = 0.1
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#3498db', label='Majority',
               alpha=0.5, edgecolors='k', s=20)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#e74c3c', label='Minority',
               alpha=0.8, edgecolors='k', s=40)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

X_db, y_db = make_classification(
    n_samples=500, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, weights=[0.9, 0.1], flip_y=0.01,
    class_sep=1.0, random_state=42
)

smote_db = SMOTE(random_state=42)
X_db_sm, y_db_sm = smote_db.fit_resample(X_db, y_db)

lr_db = LogisticRegression(max_iter=1000, random_state=42)
lr_db.fit(X_db, y_db)

lr_db_sm = LogisticRegression(max_iter=1000, random_state=42)
lr_db_sm.fit(X_db_sm, y_db_sm)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_decision_boundary(X_db, y_db, lr_db, axes[0],
                       'Decision Boundary WITHOUT SMOTE')
plot_decision_boundary(X_db_sm, y_db_sm, lr_db_sm, axes[1],
                       'Decision Boundary WITH SMOTE')

plt.tight_layout()
plt.savefig('imgs/decision_boundary_smote.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary: When to Use What

| Scenario | Recommended Approach |
|----------|---------------------|
| Mild imbalance (90:10) | Class weights or threshold tuning |
| Moderate imbalance (95:5) | SMOTE, SMOTE+Tomek, or Balanced RF |
| Severe imbalance (99:1) | Anomaly detection + ensemble |
| Large dataset | Random undersampling (fast) |
| Small dataset | SMOTE (avoids losing data) |
| Noisy data | SMOTE+Tomek or ENN (cleaning helps) |
| Interpretability critical | Class-weighted logistic regression |
| Best predictive performance | Balanced Random Forest |

**Always use multiple metrics** — never rely on accuracy alone!

## 10. Practice Exercises

### Exercise 1
Create an imbalanced dataset with ratio 98:2. Compare the F1 score of a default Random Forest vs one with `class_weight='balanced'`.

### Exercise 2
Using the dataset from above, find the optimal decision threshold that maximizes the Matthews Correlation Coefficient (MCC) instead of F1. How does the threshold differ?

### Exercise 3
Implement a 5-fold cross-validation pipeline comparing SMOTE, ADASYN, and Borderline-SMOTE on an imbalanced dataset. Which performs best in terms of PR-AUC?

### Exercise 4
Create a cost-sensitive Random Forest by passing different `class_weight` dictionaries. Vary the minority weight from 1 to 100 and plot how precision and recall trade off.

### Exercise 5
Load a real-world imbalanced dataset (e.g., credit card fraud from Kaggle, or use `fetch_openml(name='credit-g')`). Build a complete pipeline with SMOTE, threshold tuning, and evaluate using MCC. Compare 3 different classifiers.